# Notebook 092: FinOps and DBU Cost Calculation Using system.billing.usage System Tables

**Module:** Module 09: Databricks Platform, Unity Catalog, Security & FinOps  
**Difficulty:** `Advanced`  
**Databricks Masterclass Reference:** Video 15: Job Run Cost Calculation in Databricks with SQL  

---

## 1. Business Scenario & Real-World Context
A Director of Data Engineering needs to track monthly Databricks DBU (Databricks Unit) consumption across business units (Finance, Marketing, ML). The FinOps team must identify expensive rogue jobs, untagged clusters, and cost spikes.

## 2. Technical Requirement & Problem Statement
Query Databricks Unity Catalog system billing tables: analyze `system.billing.usage`, compute DBU consumption by job, cluster, and user, calculate estimated dollar expenditures, and build an automated FinOps cost anomaly detection query in PySpark SQL.

## 3. Spark & Databricks Architecture Notes
Databricks system tables (`system.billing.usage`, `system.compute.clusters`) provide historical operational telemetry. The `system.billing.usage` table records DBU consumption per SKU, account, workspace, cluster, and custom billing tags every hour.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_092_FinOps_and_DBU_Cost_Calculation_Using_system.billing.usage_System_Tables")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Mocking system.billing.usage Telemetry Data
Create representative billing usage records.

In [ ]:
usage_records = [
    ("WS-1", "JOB-101", "Nightly Ingestion", "ALL_PURPOSE", 45.2, "Finance", "2023-11-20"),
    ("WS-1", "JOB-102", "Customer 360", "JOBS", 12.8, "Marketing", "2023-11-20"),
    ("WS-1", "JOB-103", "Fraud Model Training", "DLT", 65.0, "Fraud", "2023-11-20"),
    ("WS-1", None, "Ad-Hoc Data Science", "ALL_PURPOSE", 110.5, "DataScience", "2023-11-20"),
    ("WS-1", "JOB-101", "Nightly Ingestion", "ALL_PURPOSE", 48.0, "Finance", "2023-11-21")
]

schema = "workspace_id STRING, job_id STRING, workload_name STRING, sku_name STRING, dbus DOUBLE, department STRING, usage_date STRING"
df_billing = spark.createDataFrame(usage_records, schema)
df_billing.createOrReplaceTempView("system_billing_usage")
df_billing.show()

## 6. FinOps DBU and Dollar Cost Calculation by Department
Aggregate DBU consumption and apply average contract DBU rate ($0.30/DBU).

In [ ]:
finops_query = """
SELECT 
    department,
    sku_name,
    COUNT(*) AS run_events,
    ROUND(SUM(dbus), 2) AS total_dbus,
    ROUND(SUM(dbus) * 0.30, 2) AS estimated_cost_usd
FROM system_billing_usage
GROUP BY department, sku_name
ORDER BY total_dbus DESC
"""

df_finops = spark.sql(finops_query)
print("Departmental FinOps Cost Report:")
df_finops.show(truncate=False)

## 7. Detecting Cost Anomalies and Untagged Workloads
Identify interactive all-purpose cluster usage that should run as scheduled jobs.

In [ ]:
anomaly_query = """
SELECT 
    workload_name,
    SUM(dbus) AS total_dbus,
    ROUND(SUM(dbus) * 0.30, 2) AS cost_usd,
    'WARNING: High All-Purpose Compute! Convert to Automated Jobs Cluster to save 60%' AS recommendation
FROM system_billing_usage
WHERE sku_name = 'ALL_PURPOSE' AND job_id IS NULL
GROUP BY workload_name
"""

spark.sql(anomaly_query).show(truncate=False)

## 9. Key Architectural Takeaways
- **`system.billing.usage` provides granular operational visibility into DBU spend across workspaces.**
- **Automated Jobs compute is ~60% cheaper than interactive All-Purpose compute for scheduled pipelines.**
- **Enforce custom cluster tags (e.g. `Department`, `CostCenter`, `Environment`) to attribute costs accurately.**

## 10. Senior Data Engineering Interview Questions
1. What is a DBU (Databricks Unit) and how is it calculated?
2. What is the cost difference between All-Purpose Compute and Jobs Compute in Databricks?
3. How do you query `system.billing.usage` to identify untagged compute resources?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
